In [23]:
from datasets import load_dataset
import os
import pandas as pd

Para cada linha:
1- Transformar label em df
2- Calcular a media de todos os especialistas para cada emoção
3- Pegar apenas as 27 emoções
4- Discretizar a label por um limiar (binarizar, 0 ou 1 para facilitar) - pode ser um parâmetro de teste
5- Ai o modelo vai prever quais emoções/expressoes da lista que tem, e vamos por 1 nas que foram citadas e 0 nas outras - assim vamos ter um limiar.

In [24]:
ds = load_dataset("laion/emonet-face-hq", split="train")


In [25]:
# import json

# def extract_emotion_name(full_key):
#     """
#     Extracts only the emotion name after the '|' separator.
#     Example:
#     'Negative High-Energy Emotions|Anger' -> 'Anger'
#     """
#     if "|" in full_key:
#         return full_key.split("|")[1].strip()
#     return full_key


# def parse_label(label_field):
#     """
#     Some rows have `label` as a Python list of dicts.
#     Other rows have it as a string representing that list.
#     This function normalizes both cases into a proper Python list.
#     """

#     # If it's already a Python list → return as is
#     if isinstance(label_field, list):
#         return label_field

#     # If it's a string, we must convert
#     if isinstance(label_field, str):
#         try:
#             # Try JSON decoding first
#             return json.loads(label_field)
#         except json.JSONDecodeError:
#             # If JSON fails, the string uses single quotes -> use eval safely
#             return eval(label_field)

#     raise ValueError("Unexpected label format:", type(label_field))


# def compute_mean_scores(label_field):
#     parsed = parse_label(label_field)

#     human_entries = [list(entry.values())[0] for entry in parsed]

#     emotions = human_entries[0].keys()

#     mean_scores = {}

#     for full_emo_name in emotions:
#         # Extract only the part after "|"
#         emo = full_emo_name.split("|")[-1].strip()

#         values = [h[full_emo_name] for h in human_entries]
#         mean_scores[emo] = sum(values) / len(values)

#     return mean_scores

# ds_with_means = ds.map(
#     lambda row: compute_mean_scores(row["label"]),
# )

# ds_with_means
# ds_with_means[0]
# len(ds_with_means.column_names)
# ds_with_means
# def binarize_emotions(row):
#     """
#     Given a row where each emotion column contains a mean score,
#     this function sets the highest scoring emotion(s) to 1
#     and all others to 0. Ties are preserved.
#     """

#     # Select only emotion keys (float columns)
#     emotion_values = {k: v for k, v in row.items() if isinstance(v, (float, int))}

#     # Find the maximum score for this sample
#     max_score = max(emotion_values.values())

#     # Assign 1 to all emotions that match max score, else 0
#     binary_labels = {k: 1 if v == max_score else 0 for k, v in emotion_values.items()}

#     return binary_labels

# ds_binary = ds_with_means.map(
#     binarize_emotions,
#     remove_columns=["path"]
# )

# ds_binary = ds_binary.add_column("path", ds_with_means["path"])

In [30]:
import json
import ast
import pandas as pd

def parse_label_field(x):
    """
    Ensures that the 'label' field becomes a proper Python object (list of dicts),
    regardless of whether it is JSON, a Python-like string, or already a Python object.
    """
    if isinstance(x, list):
        return x  # already a decoded Python object

    if isinstance(x, str):
        x = x.strip()

        # Try parsing as valid JSON
        try:
            return json.loads(x)
        except:
            pass

        # Try parsing as Python literal (supports single quotes)
        try:
            return ast.literal_eval(x)
        except:
            pass

        raise ValueError(f"Invalid label format: {x[:100]}")

    raise TypeError(f"Invalid type ({type(x)}). Expected str or list.")


def clean_emotion_name(full_name):
    """
    Extracts only the part after the '|' character.
    Example: 'Cognitive States|Concentration' -> 'Concentration'
    """
    if "|" in full_name:
        return full_name.split("|", 1)[1].strip()
    return full_name.strip()


def compute_mean_scores(label_field):
    """
    Processes the 'label' field (string or list) and returns a dictionary
    with mean emotion scores, with cleaned emotion names.
    """
    data = parse_label_field(label_field)

    # Extract dicts from each annotator (human-1, human-2, ...)
    human_dicts = [list(h.values())[0] for h in data]

    emotions = human_dicts[0].keys()
    mean_scores = {}

    for emo in emotions:
        cleaned_name = clean_emotion_name(emo)
        vals = [h[emo] for h in human_dicts]
        mean_scores[cleaned_name] = sum(vals) / len(vals)

    return mean_scores


# Convert HF dataset to pandas
df = ds.to_pandas()

# Compute mean emotion scores
mean_scores_list = df["label"].apply(compute_mean_scores)

# Convert dict into dataframe columns
mean_scores_df = pd.DataFrame(mean_scores_list.tolist())

# Merge with original dataframe
df_final = pd.concat([df, mean_scores_df], axis=1)

df_final.head()


,path,prompt,age,ethnicity,gender,emotion,subset,label,Concentration,Confusion,...,Interest,Pleasure/Ecstasy,Teasing,Triumph,Affection,Contemplation,Contentment,Pride,Relief,Thankfulness/Gratitude
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Sou...",6,8,3,"spite, sadism, malevolence, malice, desire to ...",1,[{'human-1': {'Cognitive States and Processes|...,2.75,0.0,...,1.0,0.00,0.00,0.50,0.25,0.25,0.00,0.75,0.0,0.75
1,{'bytes': b'RIFFbR\x12\x00WEBPVP8LUR\x12\x00/\...,Very realistic high quality portrait DSLR phot...,3,9,0,genuine silliness and jesting,1,[{'human-1': {'Cognitive States and Processes|...,2.25,0.5,...,0.5,0.00,0.25,0.00,1.75,0.75,0.50,0.25,0.0,1.00
2,{'bytes': b'RIFF:\xa1\x0e\x00WEBPVP8L.\xa1\x0e...,Very realistic high quality portrait DSLR phot...,0,11,4,genuine subtle laughter and silliness,1,[{'human-1': {'Cognitive States and Processes|...,3.75,0.0,...,2.5,0.75,0.50,0.75,2.50,3.00,2.25,3.00,1.0,0.50
3,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Nor...",7,7,4,"jealousy, envy, covetousness",1,[{'human-1': {'Cognitive States and Processes|...,2.75,0.5,...,0.5,0.00,0.00,0.00,0.50,1.00,0.75,0.00,0.0,1.00
4,{'bytes': b'RIFF~\xc2\x10\x00WEBPVP8Lr\xc2\x10...,Very realistic high quality portrait DSLR phot...,7,3,0,genuine subtle playfulness and joviality,1,[{'human-1': {'Cognitive States and Processes|...,1.25,0.0,...,1.0,0.00,0.00,0.00,0.25,1.25,1.25,0.25,0.0,0.25


In [29]:
len(df_final.columns)

48

In [32]:
df_final.iloc[22]

path                                            {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...
prompt                                          an authentic, realistic closeup image of a Ame...
age                                                                                             2
ethnicity                                                                                       0
gender                                                                                          0
emotion                                              ecstasy, pleasure, bliss, rapture, Beatitude
subset                                                                                          1
label                                           [{'human-1': {'Cognitive States and Processes|...
Concentration                                                                                2.75
Confusion                                                                                     0.0
Infatuation         

In [33]:
# Identify which columns are emotion columns.
# Adjust this list if you have additional non-emotion columns
emotion_cols = [
    col for col in df_final.columns
    if col not in ["path", "prompt", "age", "ethnicity", "gender", "emotion", "subset", "label"]
]

def binarize_row_overwrite(row):
    """
    Overwrites emotion columns with binary values:
    1 for the emotion(s) with the maximum score and 0 for the rest.
    If multiple emotions share the maximum score, all are set to 1.
    """
    # Extract only emotion scores
    scores = row[emotion_cols]

    # Find maximum value for this row
    max_value = scores.max()

    # Create binary vector
    binary = (scores == max_value).astype(int)

    # Replace original values with binary ones
    row[emotion_cols] = binary

    return row


# Apply to entire dataframe
df_final = df_final.apply(binarize_row_overwrite, axis=1)

df_final.head()


,path,prompt,age,ethnicity,gender,emotion,subset,label,Concentration,Confusion,...,Interest,Pleasure/Ecstasy,Teasing,Triumph,Affection,Contemplation,Contentment,Pride,Relief,Thankfulness/Gratitude
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Sou...",6,8,3,"spite, sadism, malevolence, malice, desire to ...",1,[{'human-1': {'Cognitive States and Processes|...,1,0,...,0,0,0,0,0,0,0,0,0,0
1,{'bytes': b'RIFFbR\x12\x00WEBPVP8LUR\x12\x00/\...,Very realistic high quality portrait DSLR phot...,3,9,0,genuine silliness and jesting,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0
2,{'bytes': b'RIFF:\xa1\x0e\x00WEBPVP8L.\xa1\x0e...,Very realistic high quality portrait DSLR phot...,0,11,4,genuine subtle laughter and silliness,1,[{'human-1': {'Cognitive States and Processes|...,1,0,...,0,0,0,0,0,0,0,0,0,0
3,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Nor...",7,7,4,"jealousy, envy, covetousness",1,[{'human-1': {'Cognitive States and Processes|...,1,0,...,0,0,0,0,0,0,0,0,0,0
4,{'bytes': b'RIFF~\xc2\x10\x00WEBPVP8Lr\xc2\x10...,Very realistic high quality portrait DSLR phot...,7,3,0,genuine subtle playfulness and joviality,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
df_final.columns

Index(['path', 'prompt', 'age', 'ethnicity', 'gender', 'emotion', 'subset',
       'label', 'Concentration', 'Confusion', 'Infatuation', 'Longing',
       'Sexual Lust', 'Anger', 'Disgust', 'Distress', 'Fear',
       'Impatience and Irritability', 'Malevolence/Malice', 'Bitterness',
       'Contempt', 'Disappointment', 'Doubt', 'Embarrassment',
       'Emotional Numbness', 'Helplessness', 'Jealousy & Envy', 'Sadness',
       'Shame', 'Fatigue/Exhaustion',
       'Intoxication/Altered States of Consciousness', 'Pain', 'Sourness',
       'Amusement', 'Astonishment/Surprise', 'Awe', 'Elation',
       'Hope/Enthusiasm/Optimism', 'Interest', 'Pleasure/Ecstasy', 'Teasing',
       'Triumph', 'Affection', 'Contemplation', 'Contentment', 'Pride',
       'Relief', 'Thankfulness/Gratitude'],
      dtype='object')

In [40]:
emotion_columns = [
    'path', 
    'prompt', 
    'age', 
    'ethnicity', 
    'gender', 
    'emotion', 
    'subset',
    'label',
    "Amusement",
    "Anger",
    "Awe",
    "Concentration",  # <-- NOT FOUND in dataset (não existe!)
    "Confusion",
    "Contemplation",
    "Contempt",
    "Contentment",
    "Longing",                   # Desire
    "Disappointment",
    "Disgust",
    "Distress",
    "Doubt",
    "Pleasure/Ecstasy",          # Ecstasy
    "Elation",
    "Embarrassment",
    "Fear",
    "Interest",
    "Infatuation",               # Love
    "Pain",
    "Pride",
    "Relief",
    "Sadness",
    "Shame",
    "Astonishment/Surprise",     # Surprise
    "Affection",                 # Sympathy
    "Triumph"
]

df_emo = df_final[emotion_columns]

In [41]:
df_emo.columns

Index(['path', 'prompt', 'age', 'ethnicity', 'gender', 'emotion', 'subset',
       'label', 'Amusement', 'Anger', 'Awe', 'Concentration', 'Confusion',
       'Contemplation', 'Contempt', 'Contentment', 'Longing', 'Disappointment',
       'Disgust', 'Distress', 'Doubt', 'Pleasure/Ecstasy', 'Elation',
       'Embarrassment', 'Fear', 'Interest', 'Infatuation', 'Pain', 'Pride',
       'Relief', 'Sadness', 'Shame', 'Astonishment/Surprise', 'Affection',
       'Triumph'],
      dtype='object')

In [43]:
only_emo = ['Amusement', 'Anger', 'Awe', 'Concentration', 'Confusion',
       'Contemplation', 'Contempt', 'Contentment', 'Longing', 'Disappointment',
       'Disgust', 'Distress', 'Doubt', 'Pleasure/Ecstasy', 'Elation',
       'Embarrassment', 'Fear', 'Interest', 'Infatuation', 'Pain', 'Pride',
       'Relief', 'Sadness', 'Shame', 'Astonishment/Surprise', 'Affection',
       'Triumph']

In [47]:
df_emo_final = df_emo[df_emo[only_emo].sum(axis=1) > 0]

In [48]:
len(df_emo_final)

2176

In [50]:
(df_emo_final[only_emo] == 0).sum()

Amusement                2102
Anger                    2086
Awe                      2161
Concentration            1372
Confusion                2131
Contemplation            1953
Contempt                 2146
Contentment              2012
Longing                  2148
Disappointment           2105
Disgust                  2171
Distress                 2104
Doubt                    2102
Pleasure/Ecstasy         2170
Elation                  2087
Embarrassment            2173
Fear                     2152
Interest                 2145
Infatuation              2057
Pain                     2168
Pride                    2102
Relief                   2164
Sadness                  2028
Shame                    2173
Astonishment/Surprise    2099
Affection                1964
Triumph                  2172
dtype: int64

In [51]:
df_emo_final

,path,prompt,age,ethnicity,gender,emotion,subset,label,Amusement,Anger,...,Interest,Infatuation,Pain,Pride,Relief,Sadness,Shame,Astonishment/Surprise,Affection,Triumph
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Sou...",6,8,3,"spite, sadism, malevolence, malice, desire to ...",1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0
1,{'bytes': b'RIFFbR\x12\x00WEBPVP8LUR\x12\x00/\...,Very realistic high quality portrait DSLR phot...,3,9,0,genuine silliness and jesting,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,1,0,0,0,0
2,{'bytes': b'RIFF:\xa1\x0e\x00WEBPVP8L.\xa1\x0e...,Very realistic high quality portrait DSLR phot...,0,11,4,genuine subtle laughter and silliness,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,1,0,0,0,0,0,0,0,0
3,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Nor...",7,7,4,"jealousy, envy, covetousness",1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0
4,{'bytes': b'RIFF~\xc2\x10\x00WEBPVP8Lr\xc2\x10...,Very realistic high quality portrait DSLR phot...,7,3,0,genuine subtle playfulness and joviality,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2494,"{'bytes': b'RIFF ""\x14\x00WEBPVP8L\x13""\x14\x0...",Very realistic high quality portrait DSLR phot...,6,8,0,genuine laughter and joviality,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0
2495,"{'bytes': b'RIFF,5\x10\x00WEBPVP8L 5\x10\x00/\...",Very realistic high quality portrait DSLR phot...,1,11,4,genuine amusement and mirth,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0
2496,{'bytes': b'RIFF\x0c\xc4\x11\x00WEBPVP8L\x00\x...,Very realistic high quality portrait DSLR phot...,3,8,0,genuine subtle amusement and joviality,1,[{'human-1': {'Cognitive States and Processes|...,0,1,...,0,0,0,0,0,0,0,0,0,0
2497,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Mid...",7,5,0,"sexual lust, carnal desire, lust, feeling horn...",1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,1,0,0,0,0


In [52]:
df_emo_final.iloc[22]

path                     {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...
prompt                   a closeup sharp focused photo of a Native Hawa...
age                                                                      7
ethnicity                                                                6
gender                                                                   0
emotion                  mild being drunk, stupor, intoxication, disori...
subset                                                                   1
label                    [{'human-1': {'Cognitive States and Processes|...
Amusement                                                                0
Anger                                                                    0
Awe                                                                      0
Concentration                                                            0
Confusion                                                                0
Contemplation            